# Catchment Hydrology, MSc course
## Lecture 2 (Europe edition): The runoff ratio of European catchments
*Wouter R. Berghuijs*

This notebook is a European counterpart to `Lecture_2_RunoffRatio.ipynb`, which used the
[CAMELS](https://ral.ucar.edu/solutions/products/camels) dataset for the contiguous United States.
Here we use **EStreams**, a pan-European catalogue of streamflow, hydro-climatic signatures and
landscape descriptors for **17,130 gauged catchments in 41 countries**, spanning roughly 1900–2022
(Nascimento et al., 2024, *Scientific Data*). See the *References* section at the end of this notebook
for the full citation and links to the complete dataset.

The tables bundled in `data/` are the *summary* parts of EStreams: one row per catchment
(`basin_id`) with pre-computed hydro-climatic signatures and static landscape attributes. The full
release also contains daily streamflow and meteorological time series (one file per catchment,
too large to ship inside a Binder repository) — those are available from the archived dataset
linked in the references.

### The runoff ratio

The **runoff ratio** is the fraction of precipitation that leaves a catchment as streamflow:

$$RR = \frac{Q}{P}$$

where $Q$ is mean streamflow and $P$ is mean precipitation (both usually expressed as a long-term
average depth per unit time, e.g. mm/day or mm/year). It is one of the simplest and most
informative catchment "fingerprints": it tells you, on average, what happens to the water that
falls on a catchment, and it is the starting point of the Budyko framework, which relates the
runoff ratio (or its complement, the evaporative index) primarily to aridity.

**Your task:** using the interactive tools below, explore how the runoff ratio of European
catchments varies geographically, and which climatic and landscape characteristics it relates to
most strongly. Work through the numbered questions as you go — they are meant to be discussed, not
answered in one line.

A quick note on data quality: EStreams is compiled from dozens of national data providers with
very different data models, so — like any real, large observational dataset — it contains a small
number of implausible values (e.g. a handful of catchments with a "runoff ratio" of several
thousand, almost certainly caused by unit or area errors upstream in the data chain). The cell
below removes the small number of physically impossible values ($RR<0$ or $RR>5$) so the plots are
readable, but leaves everything else untouched. Keep this in mind: real datasets need this kind of
sanity-checking, and *which* threshold to use is itself a judgment call worth discussing.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ipywidgets as widgets
from ipywidgets import interact

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 6)

# ---------------------------------------------------------------------------
# Load the EStreams tables (all indexed by the catchment identifier basin_id)
# ---------------------------------------------------------------------------
signatures = pd.read_csv('data/estreams_hydrometeo_signatures.csv')
gauges     = pd.read_csv('data/estreams_gauging_stations.csv')
topography = pd.read_csv('data/estreams_topography_attributes.csv')
soil       = pd.read_csv('data/estreams_soil_attributes.csv')
vegetation = pd.read_csv('data/estreams_vegetation_attributes.csv')
hydrology  = pd.read_csv('data/estreams_hydrology_attributes.csv')
geology    = pd.read_csv('data/estreams_geology_attributes.csv').drop(columns=['lit_dom'])
snowcover  = pd.read_csv('data/estreams_snowcover_attributes.csv')
met_dens   = pd.read_csv('data/estreams_meteorology_density.csv')

# Data-quality step (see markdown cell above): a runoff ratio outside [0, 5] is not physically
# plausible for these catchments and almost certainly reflects an error upstream in the data chain.
signatures.loc[(signatures['q_runoff_ratio'] < 0) | (signatures['q_runoff_ratio'] > 5),
               'q_runoff_ratio'] = np.nan

# Keep only the compact, generally useful columns from the gauging-station metadata
gauges_small = gauges[['basin_id', 'gauge_country', 'lon', 'lat', 'elevation', 'area_estreams']]

# Merge everything into one wide table, one row per catchment
merged = (signatures
          .merge(gauges_small, on='basin_id', how='left')
          .merge(topography,   on='basin_id', how='left')
          .merge(soil,         on='basin_id', how='left')
          .merge(vegetation,   on='basin_id', how='left')
          .merge(hydrology,    on='basin_id', how='left')
          .merge(geology,      on='basin_id', how='left')
          .merge(snowcover,    on='basin_id', how='left')
          .merge(met_dens,     on='basin_id', how='left')
          .set_index('basin_id'))

numeric_vars = merged.select_dtypes(include=[np.number]).columns.tolist()

print(f'{merged.shape[0]:,} catchments, {merged.shape[1]} attributes '
      f'({len(numeric_vars)} numeric), {gauges["gauge_country"].nunique()} countries')
merged.head()

17,130 catchments, 189 attributes (181 numeric), 39 countries


,q_mean,q_runoff_ratio,q_elas_Sankarasubramanian,slope_sawicz,baseflow_index,hfd_mean,hfd_std,q_5,q_95,hq_freq,...,stations_num_t_max,stations_dens_t_max,stations_num_ws_mean,stations_dens_ws_mean,stations_num_p_mean,stations_dens_p_mean,stations_num_t_min,stations_dens_t_min,stations_num_rh_mean,stations_dens_rh_mean
basin_id,,,,,,,,,,,,,,,,,,,,,
AT000001,2.824,0.727,1.266,1.505,0.760,237.600,12.858,1.029,6.607,0.077,...,28.0,0.006,22.0,0.005,22.0,0.005,28.0,0.006,17.0,0.004
AT000002,3.898,1.004,1.223,2.467,0.720,247.952,10.932,0.980,10.727,0.964,...,13.0,0.127,14.0,0.137,14.0,0.137,13.0,0.127,13.0,0.127
AT000003,0.915,0.247,1.802,0.979,0.687,233.361,27.141,0.404,2.819,6.136,...,18.0,0.034,18.0,0.034,18.0,0.034,18.0,0.034,18.0,0.034
AT000004,5.079,1.319,0.324,2.188,0.747,242.783,10.736,1.499,13.295,0.042,...,8.0,0.121,8.0,0.121,7.0,0.106,8.0,0.121,8.0,0.121
AT000005,3.319,0.806,0.820,1.967,0.756,239.207,14.642,1.064,7.692,0.233,...,10.0,0.138,10.0,0.138,10.0,0.138,10.0,0.138,10.0,0.138


In [3]:
def run_histogram(variable='q_runoff_ratio', bins=40):
    values = merged[variable].dropna()
    fig, ax = plt.subplots()
    ax.hist(values, bins=bins, color='#3E7CB1', edgecolor='white')
    ax.set_xlabel(variable)
    ax.set_ylabel('number of catchments')
    ax.set_title(f'Distribution of {variable} across {len(values):,} European catchments')
    plt.show()

    print(values.describe().to_string())
    print(f'skewness: {values.skew():.2f}')

interact(run_histogram,
         variable=widgets.Dropdown(options=numeric_vars, value='q_runoff_ratio',
                                    description='variable:'),
         bins=widgets.IntSlider(min=5, max=150, step=5, value=40, description='bins:'));

interactive(children=(Dropdown(description='variable:', index=1, options=('q_mean', 'q_runoff_ratio', 'q_elas_…

### Questions

**Q1.** What is a typical (e.g. median) runoff ratio for European catchments? How does it compare
to what you know (or can look up) about the CAMELS/US catchments in the original lecture?

**Q2.** Around 1,000 catchments have a runoff ratio greater than 1 (i.e. more water leaves the
catchment than falls on it, on average). Physically, how is that possible? Think about where these
catchments are located (try the map tool below) and what that suggests.

**Q3.** The histogram tool lets you look at the distribution of *any* variable, not just the
runoff ratio. Compare the shape of `aridity` and `frac_snow` — which is more symmetric, and why
might that be?

**Q4.** The data-cleaning step above removed catchments with a runoff ratio above 5. Try changing
that threshold in the code cell (e.g. to 2, or to 20) and re-run. How sensitive are the summary
statistics (mean, median, std) to this choice? Which statistic is most robust to a few extreme
values, and why?

**Q5.** In the scatter tool below you can compute both the Pearson and the Spearman correlation
coefficient. What is the conceptual difference between the two? When would you trust Spearman more
than Pearson for a variable like `q_runoff_ratio`?

In [5]:
def run_scatter(x='aridity', y='q_runoff_ratio', logx=False, logy=False, fit_line=True):
    data = merged[[x, y]].dropna()
    if logx:
        data = data[data[x] > 0]
    if logy:
        data = data[data[y] > 0]

    fig, ax = plt.subplots()
    ax.scatter(data[x], data[y], s=8, alpha=0.35, color='#3E7CB1', edgecolor='none')

    if fit_line and len(data) > 2:
        xs = np.linspace(data[x].min(), data[x].max(), 100)
        coeffs = np.polyfit(data[x], data[y], 1)
        ax.plot(xs, np.polyval(coeffs, xs), color='black', linewidth=1.5)

    if logx:
        ax.set_xscale('log')
    if logy:
        ax.set_yscale('log')

    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'n = {len(data):,} catchments')
    plt.show()

    rho, p_s = stats.spearmanr(data[x], data[y])
    r, p_p = stats.pearsonr(data[x], data[y])
    print(f'Spearman r = {rho:.3f}  (p = {p_s:.1e})')
    print(f'Pearson  r = {r:.3f}  (p = {p_p:.1e})')

interact(run_scatter,
         x=widgets.Dropdown(options=numeric_vars, value='aridity', description='x:'),
         y=widgets.Dropdown(options=numeric_vars, value='q_runoff_ratio', description='y:'),
         logx=widgets.Checkbox(value=False, description='log-scale x'),
         logy=widgets.Checkbox(value=False, description='log-scale y'),
         fit_line=widgets.Checkbox(value=True, description='linear fit'));

interactive(children=(Dropdown(description='x:', index=16, options=('q_mean', 'q_runoff_ratio', 'q_elas_Sankar…

**Q6.** Using the scatter tool, explore which catchment characteristics relate most strongly to
`q_runoff_ratio` — try `aridity`, `p_mean`, `frac_snow`, `ele_mt_mean`, `slp_dg_mean`,
`soil_tawc_mean`, `baseflow_index`, and a few of your own choosing. Which single variable explains
the most variance? Does this match what the Budyko framework predicts (that aridity should be the
dominant control)? Which variables add information *beyond* aridity, and can you give a physical
reason for each?

In [7]:
try:
    import geopandas as gpd
    europe = gpd.read_file(
        'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/'
        'ne_110m_admin_0_countries.geojson'
    )
    europe = europe.cx[-25:45, 33:72]  # crop to roughly the EStreams domain
except Exception:
    europe = None  # no internet available right now — the map still works, just without borders

countries = ['all'] + sorted(merged['gauge_country'].dropna().unique().tolist())

def map_maker(variable='q_runoff_ratio', country='all', vmin_pct=5, vmax_pct=95):
    subset = merged if country == 'all' else merged[merged['gauge_country'] == country]
    data = subset[['lon', 'lat', variable]].dropna()

    fig, ax = plt.subplots(figsize=(9, 9))
    if europe is not None:
        europe.boundary.plot(ax=ax, color='lightgray', linewidth=0.5, zorder=0)

    vmin, vmax = np.percentile(data[variable], [vmin_pct, vmax_pct])
    sca = ax.scatter(data['lon'], data['lat'], c=data[variable], cmap='viridis',
                      vmin=vmin, vmax=vmax, s=6, zorder=1)
    plt.colorbar(sca, ax=ax, label=variable, shrink=0.75)
    ax.set_xlabel('longitude')
    ax.set_ylabel('latitude')
    ax.set_aspect('equal')
    ax.set_title(f'{variable} — {len(data):,} catchments'
                 + ('' if country == 'all' else f' ({country})'))
    plt.show()

interact(map_maker,
         variable=widgets.Dropdown(options=numeric_vars, value='q_runoff_ratio',
                                    description='variable:'),
         country=widgets.Dropdown(options=countries, value='all', description='country:'),
         vmin_pct=widgets.IntSlider(min=0, max=49, step=1, value=5, description='low pct:'),
         vmax_pct=widgets.IntSlider(min=51, max=100, step=1, value=95, description='high pct:'));

interactive(children=(Dropdown(description='variable:', index=1, options=('q_mean', 'q_runoff_ratio', 'q_elas_…

In [8]:
core_vars = ['q_runoff_ratio', 'q_mean', 'baseflow_index', 'aridity', 'p_mean', 'pet_mean',
             'p_seasonality', 'frac_snow', 'ele_mt_mean', 'slp_dg_mean', 'strm_dens',
             'soil_tawc_mean', 'root_dep_mean', 'lai_mean', 'ndvi_mean', 'sno_cov_mean']
core_vars = [v for v in core_vars if v in merged.columns]

def correlation_overview(method='spearman'):
    corr = merged[core_vars].corr(method=method)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title(f'{method.capitalize()} correlation between core EStreams variables')
    plt.show()

interact(correlation_overview,
         method=widgets.Dropdown(options=['pearson', 'spearman'], value='spearman',
                                  description='method:'));

# Note: this heatmap deliberately uses a curated subset of ~15 variables rather than the full
# ~180-column table — with this many catchments and attributes, a full correlation matrix is both
# unreadable and slow to recompute interactively. Feel free to edit `core_vars` above to add or
# remove variables you're interested in.

interactive(children=(Dropdown(description='method:', index=1, options=('pearson', 'spearman'), value='spearma…

### References

Nascimento, T. V. M., Rudlang, J., Höge, M., van der Ent, R., Chappon, M., Seibert, J.,
Hrachowitz, M., & Fenicia, F. (2024). EStreams: An integrated dataset and catalogue of streamflow,
hydro-climatic and landscape variables for Europe. *Scientific Data*, 11, 879.
https://doi.org/10.1038/s41597-024-03706-1

Dataset (Zenodo): https://doi.org/10.5281/zenodo.13154470

Code / tools (Zenodo, mirrored on GitHub at [thiagovmdon/EStreams](https://github.com/thiagovmdon/EStreams)):
https://doi.org/10.5281/zenodo.13255133

The tables used in this notebook are a subset of the full EStreams release (static and temporal
landscape attributes, hydro-climatic signatures, and gauging-station metadata). The full release
also includes catchment boundary shapefiles and per-catchment daily streamflow and meteorological
time series — see the Zenodo dataset above for the complete archive, and please cite the paper and
dataset (and, where relevant, the original national data providers listed in
`streamflow_gauges/estreams_streamflow_catalogue.csv`) in any derived work.

Compare with the original US example: [CatchmentHydro_Lecture2_RunoffRatio](https://github.com/wberghuijs/CatchmentHydro_Lecture2_RunoffRatio)
(Addor et al., 2017; Newman et al., 2015 — CAMELS dataset).